# LGBM with random split for early stopping

**Edits by Eric Antoine Scuccimarra** - This is a fork of https://www.kaggle.com/mlisovyi/feature-engineering-lightgbm-with-f1-macro, by Misha Losvyi, with a few changes:

* The LightGBM models have been replaced with XGBoost and the code has been updated accordingly.
* I am also fitting VotingClassifiers of RandomForests and ensembling the results of the XGBs with the RFs.
* Some additional features have been added.
* Some features which were previously dropped have been retained.
* Some of the code has been reorganized.
* Rather than splitting the data once and using the validation data for the LGBM early stopping, I split the data during the training so the entire training set can be trained on. I found that this works better than a k-fold split in this case.

Some additional features were taken from:
https://www.kaggle.com/kuriyaman1002/reduce-features-140-84-keeping-f1-score, by Kuriyaman.

**Notes from Original Kernel (edited by EAS)**:

This kernel closely follows https://www.kaggle.com/mlisovyi/lightgbm-hyperoptimisation-with-f1-macro, but instead of running hyperparameter optimisation it uses optimal values from that kernel and thus runs faster.

Several key points:

* **This kernel runs training on the heads of households only** (after exttracting aggregates over households). This follows the announced scoring strategy: *Note that ONLY the heads of household are used in scoring. All household members are included in test + the sample submission, but only heads of households are scored*. (from the data description). However, at the moment it seems that evaluation depends also on non-head household members, see https://www.kaggle.com/c/costa-rican-household-poverty-prediction/discussion/61403#360115. In practice, ful prediction gives -0.4 PLB score, while replacing all non-head entries with class 1 leads to a drop down to ~0.2 PLB score
* **It seems to be very important to balance class frequencies**. Without balancing a trained model gives ~0.39 PLB / ~0.43 local test, while adding balancing leads to ~0.42 PLB / 0.47 local test. One can do it by hand, one can achieve it by undersampling. But the simplest (and more powerful compared to undersampling) is to set `class_weight='balanced'` in the LightGBM model constructor in sklearn API.
* **This kernel uses macro F1 score to early stopping in training.** This is done to align with the scoring strategy.
* Categoricals are turned into numbers with proper mapping instead of blind label encoding.
* **OHE if reversed into label encoding, as it is easier to disgest for a tree model**. This trick would be harmful for non-tree models, so be careful.
* **idhogar is NOT used in training**. The only way it could have any info would be if there is a data leak. We are fighting with poverty here- exploiting leaks will not reduce poverty in anyway :)
* **There are aggregations done within households and new features are hand-crafted.** Note, that there are not so many features that can be aggregated, as most are already quoted on household level.
* **A voting classifier is used to average over several LightGBM models**

In [15]:
import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import f1_score
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.ensemble import VotingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.utils import class_weight

import warnings
warnings.filterwarnings('ignore')

The following categorical mapping originates from [this kernel](https://www.kaggle.com/mlisovyi/categorical-variables-encoding-function).

In [16]:
from sklearn.preprocessing import LabelEncoder

def encode_data(df):
    df['idhogar'] = LabelEncoder().fit_transform(df['idhogar'])

def feature_importance(forest, X_train, display_results=True):
    ranked_list = []
    zero_features = []

    importances = forest.feature_importances_

    indices = np.argsort(importances)[::-1]

    if display_results:
        print('Feature ranking:')

    for f in range(X_train.shape[1]):
        if display_results:
            print('%d. feature %d (%f)' % (f + 1, indices[f], importances[indices[f]]) + ' - ' + X_train.columns[indices[f]])

        ranked_list.append(X_train.columns[indices[f]])

        if importances[indices[f]] == 0.0:
            zero_features.append(X_train.columns[indices[f]])
    
    return ranked_list, zero_features

**There is also feature engineering magic happening here:**

In [17]:
def do_features(df):
    feats_div = [('children_fraction', 'r4t1', 'r4t3'),
                 ('working_man_fraction', 'r4h2', 'r4t3'),
                 ('all_man_fraction', 'r4h3', 'r4t3'),
                 ('human_density', 'tamviv', 'rooms'),
                 ('human_bed_density', 'tamviv', 'bedrooms'),
                 ('rent_per_person', 'v2a1', 'r4t3'),
                 ('rent_per_room', 'v2a1', 'rooms'),
                 ('mobile_density', 'qmobilephone', 'r4t3'),
                 ('tablet_density', 'v18q1', 'r4t3'),
                 ('mobile_adult_density', 'qmobilephone', 'r4t2'),
                 ('tablet_adult_density', 'v18q1', 'r4t2'),
                ]
    
    feats_sub = [('people_not_living', 'tamhog', 'tamviv'),
                 ('people_weird_stat', 'tamhog', 'r4t3')]

    for f_new, f1, f2 in feats_div:
        df['fe_' + f_new] = (df[f1] / df[f2]).astype(np.float32)
    for f_new, f1, f2 in feats_sub:
        df['fe_' + f_new] = (df[f1] - df[f2]).astype(np.float32)
    
    aggs_num = {'age': ['min', 'max', 'mean'],
                'escolari': ['min', 'max', 'mean']}
    
    aggs_cat = {'dis': ['mean']}
    for s_ in ['estadocivil1', 'parentesco', 'instlevel']:
        for f_ in [f_ for f_ in df.columns if f_.startswith(s_)]:
            aggs_cat[f_] = ['mean', 'count']
    
    for name_, df_ in [('18', df.query('age >= 18'))]:
        df_agg = df_.groupby('idhogar').agg({**aggs_num, **aggs_cat}).astype(np.float32)
        df_agg.columns = pd.Index(['agg' + name_ + '_' + e[0] + '_' + e[1].upper() for e in df_agg.columns.tolist()])
        df = df.join(df_agg, how='left', on='idhogar')
        del df_agg
    
    df.drop(['Id'], axis=1, inplace=True)

    return df

In [18]:
def convert_OHE2LE(df):
    tmp_df = df.copy(deep=True)
    for s_ in ['pared', 'piso', 'techo', 'abastagua', 'sanitario', 'energcocinar', 'elimbasu', 'epared', 'etecho', 'eviv', 'estadocivil1', 'parentesco', 'instlevel', 'lugar', 'tipovivi','manual_elec']:
        if 'manual_' not in s_:
            cols_s_ = [f_ for f_ in df.columns if f_.startswith(s_)]
        elif 'elec' in s_:
            cols_s_ = ['public', 'planpri', 'noelec', 'coopele']
        sum_ohe = tmp_df[cols_s_].sum(axis=1).unique()

        if 0 in sum_ohe:
            print('The OHE in {} is incomplete. A new column will be added before label encoding'.format(s_))
            
            col_dummy = s_+ '_dummy'

            tmp_df[col_dummy] = (tmp_df[cols_s_].sum(axis=1) == 0).astype(np.int8)

            cols_s_.append(col_dummy)
            
            sum_ohe = tmp_df[cols_s_].sum(axis=1).unique()
            if 0 in sum_ohe:
                print('The category completion did not work')
        tmp_cat = tmp_df[cols_s_].idxmax(axis=1)
        tmp_df[s_ + '_LE'] = LabelEncoder().fit_transform(tmp_cat).astype(np.int16)
        if 'parentesco1' in cols_s_:
            cols_s_.remove('parentesco1')
        tmp_df.drop(cols_s_, axis=1, inplace=True)
    return tmp_df

# Read in the data and clean it up

In [19]:
train = pd.read_csv('./input/train.csv')
test = pd.read_csv('./input/test.csv')

test_ids = test.Id

In [20]:
def process_df(df_):
    encode_data(df_)

    return do_features(df_)

train = process_df(train)
test = process_df(test)

Clean up some missing data and convert objects to numeric.

In [21]:
train['dependency'] = np.sqrt(train['SQBdependency'])
test['dependency'] = np.sqrt(test['SQBdependency'])

train.loc[train['edjefa'] == 'no', 'edjefa'] = 0
train.loc[train['edjefe'] == 'no', 'edjefe'] = 0
test.loc[test['edjefa'] == 'no', 'edjefa'] = 0
test.loc[test['edjefe'] == 'no', 'edjefe'] = 0

train.loc[(train['edjefa'] == 'yes') & (train['parentesco1'] == 1), 'edjefa'] = train.loc[(train['edjefa'] == 'yes') & (train['parentesco1'] == 1), 'escolari']
train.loc[(train['edjefe'] == 'yes') & (train['parentesco1'] == 1), 'edjefe'] = train.loc[(train['edjefe'] == 'yes') & (train['parentesco1'] == 1), 'escolari']

train.loc[train['edjefa'] == 'yes', 'edjefa'] = 4
train.loc[train['edjefe'] == 'yes', 'edjefe'] = 4

test.loc[test['edjefa'] == 'yes', 'edjefa'] = 4
test.loc[test['edjefe'] == 'yes', 'edjefe'] = 4

train['edjefe'] = train['edjefe'].astype('int')
train['edjefa'] = train['edjefa'].astype('int')
test['edjefe'] = test['edjefe'].astype('int')
test['edjefa'] = test['edjefa'].astype('int')

train['edjef'] = np.max(train[['edjefa', 'edjefe']], axis=1)
test['edjef'] = np.max(test[['edjefa', 'edjefe']], axis=1)

train['v2a1'] = train['v2a1'].fillna(0)
test['v2a1'] = test['v2a1'].fillna(0)

test['v18q1'] = test['v18q1'].fillna(0)
train['v18q1'] = train['v18q1'].fillna(0)

train['rez_esc'] = train['rez_esc'].fillna(0)
test['rez_esc'] = test['rez_esc'].fillna(0)

train.loc[train.meaneduc.isnull(), 'meaneduc'] = 0
train.loc[train.SQBmeaned.isnull(), 'SQBmeaned'] = 0

test.loc[test.meaneduc.isnull(), 'meaneduc'] = 0
test.loc[test.SQBmeaned.isnull(), 'SQBmeaned'] = 0

train.loc[(train.v14a == 1) & (train.sanitario1 == 1) & (train.abastaguano == 0), 'v14a'] = 0
train.loc[(train.v14a == 1) & (train.sanitario1 == 1) & (train.abastaguano == 0), 'sanitario1'] = 0

test.loc[(test.v14a == 1) & (test.sanitario1 == 1) & (test.abastaguano == 0), 'v14a'] = 0
test.loc[(test.v14a == 1) & (test.sanitario1 == 1) & (test.abastaguano == 0), 'sanitario1'] = 0

In [22]:
def train_test_apply_func(train_, test_, func_):
    test_['Target'] = 0
    xx = pd.concat([train_, test_])

    xx_func = func_(xx)
    train_ = xx_func.iloc[:train_.shape[0], :]
    test_ = xx_func.iloc[train_.shape[0]:, :].drop('Target', axis=1)

    del xx, xx_func
    return train_, test_

In [23]:
train, test = train_test_apply_func(train, test, convert_OHE2LE)

The OHE in techo is incomplete. A new column will be added before label encoding
The OHE in estadocivil1 is incomplete. A new column will be added before label encoding
The OHE in instlevel is incomplete. A new column will be added before label encoding
The OHE in manual_elec is incomplete. A new column will be added before label encoding


# Geo aggregates

In [24]:
cols_2_ohe = ['eviv_LE', 'etecho_LE', 'elimbasu_LE', 'energcocinar_LE', 'sanitario_LE', 'manual_elec_LE', 'pared_LE']
cols_nums = ['age', 'meaneduc', 'dependency', 'hogar_nin', 'hogar_adul', 'hogar_mayor', 'hogar_total', 'bedrooms', 'overcrowding']

def convert_geo2aggs(df_):
    tmp_df = pd.concat([df_[(['lugar_LE', 'idhogar'] + cols_nums)], pd.get_dummies(df_[cols_2_ohe], columns=cols_2_ohe)], axis=1)

    geo_agg = tmp_df.groupby(['lugar_LE', 'idhogar']).mean().groupby('lugar_LE').mean().astype(np.float32)
    geo_agg.columns = pd.Index(['geo_' + e for e in geo_agg.columns.tolist()])

    del tmp_df
    return df_.join(geo_agg, how='left', on='lugar_LE')

train, test = train_test_apply_func(train, test, convert_geo2aggs)

In [25]:
train['num_over_18'] = 0
train['num_over_18'] = train[train.age >= 18].groupby('idhogar')['age'].transform('count')
train['num_over_18'] = train.groupby('idhogar')['num_over_18'].transform('max')
train['num_over_18'] = train['num_over_18'].fillna(0)

test['num_over_18'] = 0
test['num_over_18'] = test[test.age >= 18].groupby('idhogar')['age'].transform('count')
test['num_over_18'] = test.groupby('idhogar')['num_over_18'].transform('max')
test['num_over_18'] = test['num_over_18'].fillna(0)

def extract_features(df):
    df['bedrooms_to_rooms'] = df['bedrooms'] / df['rooms']
    df['rent_to_rooms'] = df['v2a1'] / df['rooms']
    df['tamhog_to_rooms'] = df['tamhog'] / df['rooms']
    df['r4t3_to_tamhog'] = df['r4t3'] / df['tamhog']
    df['r4t3_to_rooms'] = df['r4t3'] / df['rooms']
    df['v2a1_to_r4t3'] = df['v2a1'] / df['r4t3']
    df['v2a1_to_r4t3'] = df['v2a1'] / (df['r4t3'] - df['r4t1'])
    df['hhsize_to_rooms'] = df['hhsize'] / df['rooms']
    df['rent_to_hhsize'] = df['v2a1'] / df['hhsize']
    df['rent_to_over_18'] = df['v2a1'] / df['num_over_18']

    df.loc[df.num_over_18 == 0, 'rent_to_over_18'] = df[df.num_over_18 == 0].v2a1

extract_features(train)
extract_features(test)

In [26]:
needless_cols = ['r4t3', 'tamhog', 'tamviv', 'hhsize', 'v18q', 'v14a', 'agesq', 'mobilephone', 'female',]

instlevel_cols = [s for s in train.columns.tolist() if 'instlevel' in s]

needless_cols.extend(instlevel_cols)

train = train.drop(needless_cols, axis=1)
test = test.drop(needless_cols, axis=1)

## Split the data

We split the data by household to avoid leakeage, since rows belonging to the same household usually have the same target. Since we filter the data to only include heads of household this isn't technically necessary, but it provides an easy way to use the entire training data set if we want to do that.

Note that after splitting the data we overwrite the train data with the entire data set so we can train on all of the data. The split_data function does the same thing without overwriting the data, and is used within the training loop to (hopefully) approximate a K-Fold split.

In [27]:
def split_data(train, y, sample_weight=None, households=None, test_percentage=0.20, seed=None):
    train2 = train.copy()

    cv_hhs = np.random.choice(households, size=int(len(households) * test_percentage), replace=false)

    cv_idx = np.isin(households, cv_hhs)
    X_test = train2[cv_idx]
    y_test = y[cv_idx]

    X_train = train2[~cv_idx]
    y_train = y[~cv_idx]

    if sample_weight is not None:
        y_train_weights = sample_weight[~cv_idx]
        return X_train, y_train, X_test, y_test, y_train_weights
    
    return X_train, y_train, X_test, y_test

In [28]:
X = train.query('parentesco1==1')

y = X['Target'] - 1
X = X.drop(['Target'], axis=1)

np.random.seed(seed=None)

train2 = X.copy()

train_hhs = train2.idhogar

households = train2.idhogar.unique()
cv_hhs = np.random.choice(households, size=int(len(households) * 0.15), replace=False)

cv_idx = np.isin(train2.idhogar, cv_hhs)

X_test = train2[cv_idx]
y_test = y[cv_idx]

X_train = train2[~cv_idx]
y_train = y[~cv_idx]

X_train = train2
y_train = y

train_households = X_train.idhogar

In [29]:
y_train_weights = class_weight.compute_sample_weight('balanced', y_train, indices=None)

In [ ]:
extra_drop_features = [
    'agg18_estadocivil1_MEAN',
    'agg18_estadocivil6_COUNT',
    'agg18_estadocivil7_COUNT',
    'agg18_parentesco10_COUNT',
    'agg18_parentesco11_COUNT',
    'agg18_parentesco12_COUNT',
    'agg18_parentesco1_COUNT',
    'agg18_parentesco2_COUNT',
    'agg18_parentesco3_COUNT',
    'agg18_parentesco4_COUNT',
    'agg18_parentesco5_COUNT',
    'agg18_parentesco6_COUNT',
    'agg18_parentesco7_COUNT',
    'agg18_parentesco8_COUNT',
    'agg18_parentesco9_COUNT',
    'geo_elimbasu_LE_4',
    'geo_energcocinar_LE_1',
    'geo_energcocinar_LE_2',
    'geo_epared_LE_0',
    'geo_hogar_mayor',
    'geo_manual_elec_LE_2',
    'geo_pared_LE_3',
    'geo_pared_LE_4',

]